# 05 — Defect Classifier (auto class assignment + metrics)

Trains a small ResNet34 25-class classifier so the dashboard can auto-assign the defect class
and run the diagnostic pipeline end-to-end. Classification is easy on this dataset (paper reports
~94% with CLIP-Adapter); this is a fast ResNet baseline. Run on **Kaggle T4** or locally.

Saves: `models/classifier.pt`, `models/class_names.json`, and
`results/classification/classifier_metrics.json` (accuracy, macro-F1, per-class) for the report + dashboard.

In [ ]:
import os, sys
if os.path.isdir('/kaggle'):
    os.chdir('/kaggle/working')
    if not os.path.isdir('SteelDefectX'):
        !git clone https://github.com/d-mondal/SteelDefectX.git
    os.chdir('/kaggle/working/SteelDefectX'); 
    !git pull
    !pip install -q scikit-learn
    DATA_ROOT = '/kaggle/working/sdx_data'
    if not os.path.isdir(f'{DATA_ROOT}/train'):
        !apt-get -qq install git-lfs && git lfs install
        !git clone https://huggingface.co/datasets/Zhaosxian/SteelDefectX {DATA_ROOT}
else:
    if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')
    DATA_ROOT = 'sdx_data'
sys.path.insert(0, os.getcwd())
import torch; print('CUDA:', torch.cuda.is_available())

In [ ]:
import json, cv2, numpy as np, random
from torch.utils.data import Dataset, DataLoader

train_text = json.load(open(f'{DATA_ROOT}/train-text.json'))
val_text   = json.load(open(f'{DATA_ROOT}/val-text.json'))
class_names = sorted({e['class_name'] for e in train_text})
cls2idx = {c:i for i,c in enumerate(class_names)}
print('classes:', len(class_names))

MEAN = np.array([0.485,0.456,0.406], dtype=np.float32)
STD  = np.array([0.229,0.224,0.225], dtype=np.float32)

class DefectClsDataset(Dataset):
    def __init__(self, entries, img_dir, train=True):
        self.e = entries; self.img_dir = img_dir; self.train = train
    def __len__(self): return len(self.e)
    def __getitem__(self, i):
        en = self.e[i]
        g = cv2.imread(f'{DATA_ROOT}/{self.img_dir}/{en["image_name"]}', cv2.IMREAD_GRAYSCALE)
        g = cv2.resize(g, (256,256))
        if self.train and random.random() < 0.5: g = cv2.flip(g, 1)
        x = np.stack([g,g,g],0).astype(np.float32)/255.0
        x = (x - MEAN[:,None,None]) / STD[:,None,None]
        return torch.tensor(x), cls2idx[en['class_name']]

tl = DataLoader(DefectClsDataset(train_text,'train',True),  batch_size=32, shuffle=True,  num_workers=2)
vl = DataLoader(DefectClsDataset(val_text,  'val',  False), batch_size=32, shuffle=False, num_workers=2)
print('train', len(train_text), '| val', len(val_text))

In [ ]:
from src.segmentation.classifier import build_classifier
device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = build_classifier(len(class_names), pretrained=True).to(device)
opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
crit = torch.nn.CrossEntropyLoss()

from tqdm import tqdm
best_acc, best_state = 0.0, None
for ep in range(8):
    net.train()
    for x,y in tqdm(tl, desc=f'e{ep}', leave=False):
        x,y = x.to(device), y.to(device)
        opt.zero_grad(); loss = crit(net(x), y); loss.backward(); opt.step()
    net.eval(); correct=tot=0
    with torch.no_grad():
        for x,y in vl:
            p = net(x.to(device)).argmax(1).cpu()
            correct += (p==y).sum().item(); tot += len(y)
    acc = correct/tot; print(f'e{ep} val_acc={acc:.4f}')
    if acc>best_acc: best_acc, best_state = acc, {k:v.cpu().clone() for k,v in net.state_dict().items()}
print('best val acc:', round(best_acc,4))
net.load_state_dict(best_state)

## Compute full metrics (accuracy, macro-F1, per-class) on the held-out val set

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
net.eval(); y_true, y_pred = [], []
with torch.no_grad():
    for x,y in vl:
        p = net(x.to(device)).argmax(1).cpu().numpy()
        y_pred += p.tolist(); y_true += y.numpy().tolist()

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')
weighted_f1 = f1_score(y_true, y_pred, average='weighted')
rep = classification_report(y_true, y_pred, target_names=class_names,
                            output_dict=True, zero_division=0)
per_class = {c: {'precision': round(rep[c]['precision'],4),
                 'recall': round(rep[c]['recall'],4),
                 'f1': round(rep[c]['f1-score'],4),
                 'n': int(rep[c]['support'])} for c in class_names}

metrics = {
    'model': 'resnet34',
    'accuracy': round(acc,4),
    'macro_f1': round(macro_f1,4),
    'weighted_f1': round(weighted_f1,4),
    'n_val': len(y_true),
    'n_classes': len(class_names),
    'paper_reference': {'method': 'CLIP-Adapter', 'accuracy': 0.94,
                        'note': 'paper number for context; not an identical setup'},
    'per_class': per_class,
}
print('accuracy:', metrics['accuracy'], '| macro-F1:', metrics['macro_f1'])

In [ ]:
os.makedirs('models', exist_ok=True)
os.makedirs('results/classification', exist_ok=True)
torch.save(best_state, 'models/classifier.pt')
json.dump(class_names, open('models/class_names.json','w'))
json.dump(metrics, open('results/classification/classifier_metrics.json','w'), indent=2)
print('saved:')
print('  models/classifier.pt')
print('  models/class_names.json')
print('  results/classification/classifier_metrics.json')
if os.path.isdir('/kaggle'): print('\n-> download these from the Kaggle Output tab')